In [3]:
import numpy as np
from numpy import genfromtxt
import matplotlib.pyplot as plt
import csv
import os
import pandas as pd
import math

In [4]:
# System Parameters

accumulator_series_elements = 10*14
cell_max_voltage = 4.2 #volts
cell_min_voltage = 2.5 #volts

min_pack_voltage = cell_min_voltage*accumulator_series_elements
max_pack_voltage = cell_max_voltage*accumulator_series_elements
mid_pack_voltage = (min_pack_voltage + max_pack_voltage)/2

pmax = 850 #Watts
vout = 24 #volts
iout = pmax/vout

i_in_min = pmax/max_pack_voltage
i_in_max = pmax/min_pack_voltage

acceptable_dvout = 0.5 #volts

fsw_post = 50*10**3 #Hz post rectification
fsw_pre = fsw_post/2 #Hz pre rectification aka each phase


In [5]:
#Core Parameters
core_a_l = 175*10**-9 #Henrys
saturation_flux_density = 10500 #Gauss
core_cross_section = 262 *10**-6 #m2

In [6]:
# Transformer Calculations

calc_transformer_turns_ratio = min_pack_voltage/vout
# print(calc_transformer_turns_ratio)
actual_transformer_turns_ratio = 13

min_duty_cycle = vout/(max_pack_voltage/actual_transformer_turns_ratio)
mid_duty_cycle = vout/(mid_pack_voltage/actual_transformer_turns_ratio)
max_duty_cycle = vout/(min_pack_voltage/actual_transformer_turns_ratio)

# print(max_duty_cycle)
# print(mid_duty_cycle)
# print(min_duty_cycle)

minimum_primary_turns = (max_pack_voltage*10**8)/(4.44*10**4*fsw_pre*core_cross_section*saturation_flux_density)
# print(minimum_primary_turns)

actual_primary_turns = 26
actual_secondary_turns = 2 #per secondary coil

maximum_flux_density = (max_pack_voltage*10**8)/(4.44*10**4*fsw_pre*core_cross_section*actual_primary_turns)
# print(maximum_flux_density)
flux_density_safety_factor = saturation_flux_density/maximum_flux_density
# print(flux_density_safety_factor)

primary_self_inductance = core_a_l*actual_primary_turns**2
print(primary_self_inductance*10**6) #uH


118.30000000000001


In [7]:
# FET Calculations
# https://www.ti.com/lit/an/slyt664/slyt664.pdf?ts=1745717291353&ref_url=https%253A%252F%252Fwww.google.com%252F

ton_top = (max_duty_cycle/2)/fsw_pre
print(ton_top)
ton_bot = (1-(max_duty_cycle/2))/fsw_pre
print(ton_bot)

switching_fet_rdson = 0.01 #Ohms #TODO placeholder value
switching_fet_qg = 20*10**-9 #C #TODO placeholder value

switching_fet_conduction_losses_top = ton_top*switching_fet_rdson*i_in_max**2
switching_fet_conduction_losses_bot = ton_bot*switching_fet_rdson*i_in_max**2
#TODO consider affect of ripple in current

gate_driver_source_current = 1 #A
#TODO consider asymmetrical drive currents

switching_fet_switching_losses = max_pack_voltage*i_in_max*fsw_pre*(switching_fet_qg/gate_driver_source_current)

switching_fet_total_losses_top = switching_fet_conduction_losses_top + switching_fet_switching_losses
switching_fet_total_losses_bot = switching_fet_conduction_losses_bot + switching_fet_switching_losses

print(switching_fet_total_losses_top) #Watts
print(switching_fet_total_losses_bot) #Watts

1.782857142857143e-05
2.217142857142857e-05
0.7140010515218659
0.7140013076618076


In [8]:
# Rectifier Calculations



In [9]:
# Filter Calculations
# https://www.ti.com/lit/an/slva477b/slva477b.pdf?ts=1745747396768

acceptable_inductor_ripple_current = 0.3*iout #Amps
calculated_inductance = vout*(mid_pack_voltage-vout)/(acceptable_inductor_ripple_current*fsw_post*mid_pack_voltage)
print(calculated_inductance*10**6) #uH
actual_inductance = 47*10**-6 #H

cout = acceptable_inductor_ripple_current/(8*fsw_post*acceptable_dvout)
print(cout*10**6) #uF

42.864668255361856
53.12499999999999


In [ ]:
# Equivalent Capacitance Function Definition
# https://www.power-mag.com/pdf/feature_pdf/1387888355_Murata_Feature_Layout_1.pdf

#Returned datastructure
class EquivalentCapacitance:
    def __init__(self, capacitances, fsw, ripple_current, equivalent_capacitance, equivalent_esr, ripple_voltage):
        self.capacitances = capacitances # Will be of the form [qty, capacitance (uF), esr (mOhm), ripple current (A)] where the 4th column will only be added if a total ripple current was provided
        self.fsw = fsw
        self.ripple_current = ripple_current
        self.equivalent_capacitance = equivalent_capacitance
        self.equivalent_esr = equivalent_esr
        self.ripple_voltage = ripple_voltage

    def print(self):
        print(f"Capacitance Matrix: {self.capacitances}")
        print(f"Switching Frequency: {self.fsw} kHz")
        print(f"Ripple Current: {self.ripple_current} A")
        print(f"Equivalent Capacitance: {self.equivalent_capacitance} uF")
        print(f"Equivalent ESR: {self.equivalent_esr} mOhms")
        print(f"Ripple Voltage: {self.ripple_voltage} mV")

def equiv_capacitance(capacitances, fsw, current = 0):
    # capacitances is a 3 column, n row matrix with rows of the form [qty, capacitance (uF), esr (mOhm)]
    # fsw is the fundamental switching frequency (kHz)
    # current is optional. If provided, the return will include the ripple voltage and current per capacitor
    
    fsw = fsw * 10**3 #convert to Hz
    cs = np.array(capacitances)
    cs = cs.astype(float)
    cs[:, 1]  = cs[:, 1] * 10**-6 # Convert to F
    cs[:, 2] = cs[:, 2] * 10**-3 # Convert to Ohms
    xs = 1/(2*3.1416*fsw*cs[:, 1])
    
    cp = cs.copy()
    cp[:, 1] = cs[:, 1]/(1 + (cs[:, 2]/xs)**2)
    cp[:, 2] = (cs[:, 2]**2 + xs**2)/cs[:, 2]
    
    cpe = np.sum(cp[:, 0]*cp[:, 1])
    rpe = 1/np.sum(cp[:, 0]/cp[:, 2])

    xpe = 1/(2*3.1416*fsw*cpe)
    cse = cpe*(1 + (xpe/rpe)**2)
    rse = rpe/(1 + (rpe/xpe)**2)
    xse = 1/(2*3.1416*fsw*cse)

    dv = np.nan
    if(current != 0):
        dv = current*math.sqrt(rse**2 + xse**2)
        di = dv/np.sqrt(cs[:, 2]**2 + xs**2)
        cs = np.hstack((cs, di.reshape(-1, 1)))
        dv = dv * 10**3 # Convert to mV
    cse = cse * 10**6 # convert to uF
    rse = rse * 10**3 # convert to mOhms
    fsw = fsw * 10**-3 #convert to kHz
    return EquivalentCapacitance(cs, fsw, current if current > 0 else np.nan, cse, rse, dv)


In [106]:
# Total Efficiency

capacitances = [
                [3, 22, 4], 
                [1, 100, 8]
                ]  # qty, capacitance (uf), esr (mohm)
demo_caps = equiv_capacitance(capacitances, 200, 2)
demo_caps.print()

Capacitance Matrix: [[3.00000000e+00 2.20000000e-05 4.00000000e-03 3.40633080e-01]
 [1.00000000e+00 1.00000000e-04 8.00000000e-03 1.09858845e+00]]
Switching Frequency: 200.0 kHz
Ripple Current: 2 A
Equivalent Capacitance: 143.41358482100165 uF
Equivalent ESR: 2.7618858297945668 mOhms
Ripple Voltage: 12.396315638590496 mV
